In [6]:
import os, sys

print("Aantal paden in sys.path:", len(sys.path))
# Bepaal de parent-werkmap
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

#Loop door alle submappen en voeg ze toe aan sys.path
for dirpath, dirnames, filenames in os.walk(parent_dir):
    if dirpath not in sys.path:
        sys.path.insert(0, dirpath)

print("Aantal paden in sys.path:", len(sys.path))

Aantal paden in sys.path: 327
Aantal paden in sys.path: 327


In [7]:
from Libraries.inference_training import Configuration, ImageDataset
from Libraries.inference_training import initCudaEnvironment, createTransforms
from Libraries.inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from Libraries.inference_training import trainModel, saveModel, loadModel
import random
import sys
from pathlib import Path
print(str(Path().resolve().parents[1]))
from paths import *


C:\Users\Tomkr\Jaar2BlokD\Tygron


In [8]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

In [9]:
def createModel(trainDirectory: str, testDirectory: str, modelName: str, epochs: int, labels: list[str], augment_data: bool, save_path: str, save_interval=0, maskdata=None , model_description="default"):
    """
    Create, configure, train, and export an instance segmentation model using a configurable setup.

    This function prepares the dataset, configures the training pipeline, trains the model for the
    specified number of epochs, and exports the resulting model to both PyTorch and ONNX formats.

    Parameters:
    -----------
    trainDirectory : str
        Path to the training dataset folder.
    testDirectory : str
        Path to the testing dataset folder.
    modelName : str
        Name assigned to the trained model and saved output files.
    epochs : int
        Number of training epochs.
    labels : list[str]
        List of label names used for segmentation. For example, ['parking_spaces'].
    augment_data : bool
        Whether to apply data augmentation during training.
    save_path : str
        Directory path where the model will be saved.
    save_interval : int, optional
        Interval (in epochs) at which to save model checkpoints. Default is 0 (disabled).
    maskdata : list[float], optional
        List containing three float values:
        [scoreThreshold, maskThreshold, strideFraction] used for ONNX metadata.
        Defaults to [0.2, 0.3, 0.5] if not provided.
    model_description : str, optional
        Description to embed into the ONNX metadata. Default is "default".

    Returns:
    --------
    model : torch.nn.Module
        The trained model set to evaluation mode.
    config : Configuration
        The configuration object used for training and exporting the model.

    Notes:
    ------
    - The function prints dataset statistics and model file names for verification.
    - Legend entries are dynamically created based on the provided `labels` list.
    - If `augment_data` is True, data augmentation is applied during training using `createTransforms(True)`.
    - The function validates dataset consistency before training.
    - After training, the model is saved and exported to ONNX, and metadata is written.
    """
    if maskdata is None:
        maskdata = [0.2, 0.3, 0.5]
    config = Configuration()
    print("Device: " + str(config.device))
    config.setSaveInterval(save_interval)
    config.setSavePath(save_path)
    config.setIsCrowd(False)
    config.setDatasetPaths(trainPath=trainDirectory, testPath=testDirectory)
    config.setFilePrefix("")
    config.setModelName(modelName)
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2 + 1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.setEpochs(epochs)
    config.setOnnxInfo(producer="Tygron", description=model_description)
    config.addLegendEntry("Background", 0, "#00000000")
    i = 1
    for label in labels:
        config.addLegendEntry(label, i, "#" + ''.join([random.choice('ABCDEF0123456789') for i in range(6)]))
        i += 1

    config.setOnnxMetaData(scoreThreshold=maskdata[0],
                           maskThreshold=maskdata[1],
                           strideFraction=maskdata[2])

    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    if augment_data:
        trainingDataset = ImageDataset(config, True, imageTransforms=createTransforms(True))
        testDataset = ImageDataset(config, False, createTransforms(False))
    else:
        trainingDataset = ImageDataset(config, True, createTransforms(False))
        testDataset = ImageDataset(config, False, createTransforms(False))

    print("Train Image count: " + str(trainingDataset.__len__()))
    print("Test Image count: " + str(testDataset.__len__()))

    if not trainingDataset.validateFiles():
        print("Inconsistent training dataset ")
        trainingDataset.validateFiles()

    if not testDataset.validateFiles():
        print("Inconsistent test dataset ")
        testDataset.validateFiles()

    print("Pytorch model name " + config.getPytorchModelFileName())
    print("Onnx file name " + config.getOnnxFileName())

    model = trainModel(config, trainingDataset, testDataset)
    model.eval()

    saveModel(config, model, epoch=epochs)

    exportOnnxModel(config, model)
    writeONNXMeta(config)

    return model, config

# create model template

In [ ]:
train_directory = str(TRAIN)
test_directory = str(TEST)
modelName = "inference_2c_25cm_25e_250_maxgt.onnx"
epochs = 10
labels = ["parking_space"]
augment = False
save_path = str(MODELS_DIR)
save_Interval = 0  # model is saved in between these amount of epochs
model, config = createModel(
	train_directory,
	test_directory,
	modelName,
	epochs,
	labels,
	augment,
	save_path,
	save_Interval
)

Device: cpu
Train Image count: 400
Test Image count: 1200
Pytorch model name C:\Users\Tomkr\Jaar2BlokD\Tygron\Tygron\Modelsinference_2c_25cm_25e_250_maxgt.onnx
Onnx file name C:\Users\Tomkr\Jaar2BlokD\Tygron\Tygron\Modelsinference_2c_25cm_25e_250_maxgt.onnx.onnx
Epoch: [0]  [  0/200]  eta: 0:35:27  lr: 0.000030  loss: 4.7875 (4.7875)  loss_classifier: 1.2111 (1.2111)  loss_box_reg: 0.2115 (0.2115)  loss_mask: 2.3725 (2.3725)  loss_objectness: 0.9408 (0.9408)  loss_rpn_box_reg: 0.0516 (0.0516)  time: 10.6384  data: 0.0890
Epoch: [0]  [ 10/200]  eta: 0:32:12  lr: 0.000281  loss: 3.5366 (3.3395)  loss_classifier: 1.0131 (0.8942)  loss_box_reg: 0.1297 (0.1463)  loss_mask: 1.6764 (1.5561)  loss_objectness: 0.3960 (0.6484)  loss_rpn_box_reg: 0.0245 (0.0945)  time: 10.1716  data: 0.0746
Epoch: [0]  [ 20/200]  eta: 0:30:55  lr: 0.000532  loss: 2.0056 (2.5908)  loss_classifier: 0.3966 (0.6067)  loss_box_reg: 0.1593 (0.1685)  loss_mask: 0.8884 (1.1679)  loss_objectness: 0.3178 (0.5353)  loss_rpn

^ AP and AR converge to the results at the last epoch and don't change after with the current dataset.

In [ ]:
train_directory = str(TRAIN)
test_directory = str(TEST)
modelName = "inference_2c_25cm_15e_250_parking.onnx"  # Updated to match actual configuration
epochs = 10
labels = ["parking_space"]
augment = False
save_path = str(MODELS_DIR)
save_Interval = 0  # model is saved in between these amount of epochs

# Add error handling to help diagnose issues
try:
    model, config = createModel(
        train_directory,
        test_directory,
        modelName,
        epochs,
        labels,
        augment,
        save_path,
        save_Interval,
        maskdata=[0.2, 0.3, 0.5],  # Explicitly set mask parameters
        model_description="Parking space detection model - 250px at 25cm resolution"
    )
    print("Model created successfully!")
except Exception as e:
    print(f"Error creating model: {str(e)}")
    # Print more details about the environment
    print(f"Train directory exists: {os.path.exists(train_directory)}")
    print(f"Test directory exists: {os.path.exists(test_directory)}")
    print(f"Save path exists: {os.path.exists(save_path)}")
    raise  # Re-raise the exception to see the full traceback

Device: cpu
Train Image count: 50
Test Image count: 400
Pytorch model name C:\Users\Tomkr\Jaar2BlokD\Tygron\Tygron\Modelsinference_2c_25cm_15e_250_parking.onnx
Onnx file name C:\Users\Tomkr\Jaar2BlokD\Tygron\Tygron\Modelsinference_2c_25cm_15e_250_parking.onnx.onnx
Epoch: [0]  [ 0/25]  eta: 0:09:55  lr: 0.000213  loss: 5.7175 (5.7175)  loss_classifier: 0.9686 (0.9686)  loss_box_reg: 0.0411 (0.0411)  loss_mask: 1.6997 (1.6997)  loss_objectness: 2.4718 (2.4718)  loss_rpn_box_reg: 0.5362 (0.5362)  time: 23.8093  data: 0.0614
Epoch: [0]  [10/25]  eta: 0:06:09  lr: 0.002294  loss: 2.8964 (3.0107)  loss_classifier: 0.3272 (0.5114)  loss_box_reg: 0.1166 (0.1097)  loss_mask: 1.1234 (1.2003)  loss_objectness: 0.6437 (0.9957)  loss_rpn_box_reg: 0.0606 (0.1937)  time: 24.6521  data: 0.0843
Epoch: [0]  [20/25]  eta: 0:02:02  lr: 0.004376  loss: 1.2004 (2.0698)  loss_classifier: 0.1961 (0.3408)  loss_box_reg: 0.0986 (0.0999)  loss_mask: 0.5567 (0.8690)  loss_objectness: 0.2475 (0.6158)  loss_rpn_box